In [1]:
import hdf5plugin
import h5py
import numpy as np

In [18]:
def prepare_fitburst_input_from_ynet_outputs(h5path: str, outfile: str):
    with h5py.File(h5path, "r") as h5file:
        data = h5file["input"]["x"]["data"][:]
        times = h5file["input"]["x"]["times"][:].ravel()
        freqs = h5file["input"]["x"]["freqs"][:].ravel()
        
        metadata = {
            "bad_chans": [],
            "freqs_bin0": freqs[0],
            "is_dedispersed": False,
            "num_freq": freqs.size,
            "num_time": times.size,
            "times_bin0": 0.0,
            "res_freq": freqs[1] - freqs[0],
            "res_time": times[1] - times[0],
        }
        
        toa = h5file["output"].attrs["toa"][:] - times[0]
        dm = h5file["output"].attrs["dm"][:]
        burst_cnt = len(toa)
        params = {
            "amplitude": [0.] * burst_cnt,
            # "arrival_time": toa.tolist(),
            "arrival_time": [0.1],
            "burst_width": [0.002] * burst_cnt,
            "dm": dm.tolist(),
            "dm_index": [-2.] * burst_cnt,
            "ref_freq": [1500.] * burst_cnt,
            "scattering_index": [-4.] * burst_cnt,
            "scattering_timescale": [0.] * burst_cnt,
            "spectral_index": [0.] * burst_cnt,
            "spectral_running": [-300.] * burst_cnt,
        }
        # params = {
        #     "amplitude": [0.],
        #     "arrival_time": [0.1],
        #     "burst_width": [0.002],
        #     "dm": [570.0],
        #     "dm_index": [-2.],
        #     "ref_freq": [1500.],
        #     "scattering_index": [-4.],
        #     "scattering_timescale": [0.],
        #     "spectral_index": [0.],
        #     "spectral_running": [-300.],
        # }
        print(params)
        
        model = data.T
        model = model.astype(np.float32) * 255.0
        np.savez(outfile, data_full=model, metadata=metadata, burst_parameters=params)

In [19]:
prepare_fitburst_input_from_ynet_outputs("FRB20121102_0098.fits-chk2-new.h5", "FRB20121102_0098.npz")

{'amplitude': [0.0], 'arrival_time': [0.1], 'burst_width': [0.002], 'dm': [573.4264526367188], 'dm_index': [-2.0], 'ref_freq': [1500.0], 'scattering_index': [-4.0], 'scattering_timescale': [0.0], 'spectral_index': [0.0], 'spectral_running': [-300.0]}
